# 01 — Classical Monte Carlo Option Pricing

## Theory

Under the risk-neutral measure, the stock price at expiry follows **geometric Brownian motion (GBM)**:

$$S_T = S_0 \exp\!\left(\left(r - \tfrac{1}{2}\sigma^2\right)T + \sigma\sqrt{T}\,Z\right), \quad Z \sim \mathcal{N}(0,1)$$

The fair value of a European call option is the discounted expected payoff:

$$C = e^{-rT}\,\mathbb{E}^\mathbb{Q}\!\left[\max(S_T - K,\, 0)\right]$$

Monte Carlo approximates this expectation by averaging over $N$ independent simulated paths:

$$\hat{C}_N = e^{-rT} \cdot \frac{1}{N}\sum_{i=1}^{N} \max(S_T^{(i)} - K,\, 0)$$

By the Central Limit Theorem, the **standard error** decays as $\sigma_{\hat{C}} \propto 1/\sqrt{N}$, so to halve the error you must quadruple the sample size — a key limitation that quantum computing aims to overcome.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')

from src.classical import monte_carlo_call
from src.black_scholes import black_scholes_call

## Parameters

| Parameter | Symbol | Value |
|---|---|---|
| Initial stock price | $S_0$ | 100 |
| Strike price | $K$ | 105 |
| Risk-free rate | $r$ | 5% |
| Volatility | $\sigma$ | 20% |
| Time to expiry | $T$ | 1 year |

In [ ]:
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.2, 1.0

# Exact price for reference
bs_price = black_scholes_call(S0, K, r, sigma, T)
print(f"Black-Scholes reference price: {bs_price:.4f}")

# Single Monte Carlo run
mc_price, mc_err = monte_carlo_call(S0, K, r, sigma, T, N=100_000)
print(f"Monte Carlo (N=100,000):        {mc_price:.4f} ± {mc_err:.4f}")

## Convergence Analysis

We run the Monte Carlo pricer for increasing values of $N$ and track how the estimate and its standard error converge to the Black-Scholes price.

In [ ]:
N_values = np.logspace(2, 5, 30, dtype=int)
prices, std_errs = [], []

for N in N_values:
    p, se = monte_carlo_call(S0, K, r, sigma, T, N)
    prices.append(p)
    std_errs.append(se)

prices = np.array(prices)
std_errs = np.array(std_errs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: price estimate with ±1 std-err band
ax = axes[0]
ax.semilogx(N_values, prices, 'o-', color='steelblue', markersize=4, label='MC estimate')
ax.fill_between(N_values, prices - std_errs, prices + std_errs, alpha=0.25, color='steelblue', label='±1 std err')
ax.axhline(bs_price, color='crimson', linestyle='--', linewidth=1.5, label=f'Black-Scholes ({bs_price:.3f})')
ax.set_xlabel('Number of paths (N)')
ax.set_ylabel('Option price')
ax.set_title('Monte Carlo convergence to Black-Scholes')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Right: standard error on log-log scale — should track 1/sqrt(N)
ax = axes[1]
ax.loglog(N_values, std_errs, 'o-', color='darkorange', markersize=4, label='Std error')
ref = std_errs[0] * np.sqrt(N_values[0] / N_values)
ax.loglog(N_values, ref, '--', color='gray', label=r'$\propto 1/\sqrt{N}$ reference')
ax.set_xlabel('Number of paths (N)')
ax.set_ylabel('Standard error')
ax.set_title(r'Standard error decay: $O(1/\sqrt{N})$')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

The right panel confirms the theoretical $O(1/\sqrt{N})$ convergence rate. Quantum Amplitude Estimation achieves $O(1/M)$ where $M$ is the number of quantum oracle calls — a **quadratic speedup** over classical Monte Carlo.